In [12]:
import nltk
from nltk import CFG, ChartParser
import re



In [13]:
# 1. Load the lowercase, word-level CFG
grammar = CFG.fromstring("""
    s -> greeting query | query
    greeting -> salutation title punc | salutation title | salutation punc

    salutation -> 'hello' | 'hi' | 'dear' | 'good' 'morning'
    title -> 'cod' | 'dean'
    punc -> ','

    query -> marks_query | reg_query | waiver_query | finance_query | grad_query

    marks_query -> intro_marks noun_marks prep unit
    intro_marks -> 'i' 'am' 'inquiring' 'about' 'my' | 'i' 'am' 'reporting' 'a' | 'i' 'have' | 'my'
    noun_marks -> 'missing' 'marks' | 'missing' 'mark' | 'results'
    prep -> 'for' | 'in'
    unit -> 'unit' unit_code | unit_code
    unit_code -> 'ccs' '3104' | 'sma' '2101' | 'toc'

    reg_query -> intro_reg noun_reg prep target_reg time_frame
    intro_reg -> 'what' 'is' 'the' | 'when' 'is' 'the' | 'the'
    noun_reg -> 'deadline'
    target_reg -> 'unit' 'registration' | 'special' 'exam'
    time_frame -> 'this' 'semester' | 'now'

    waiver_query -> intro_waiver noun_waiver prep det target_waiver 'unit' | intro_waiver noun_waiver prep target_waiver
    intro_waiver -> 'i' 'would' 'like' 'to' 'request' 'a' | 'i' 'need' 'a'
    noun_waiver -> 'prerequisite' 'waiver' | 'waiver'
    det -> 'the' | 'a'
    target_waiver -> 'theory' 'of' 'computation' | 'ai'

    finance_query -> intro_finance 'the' noun_finance prep 'the' time_year
    intro_finance -> 'how' 'do' 'i' 'apply' 'for' | 'where' 'is'
    noun_finance -> 'university' 'bursary' | 'bursary'
    time_year -> 'current' 'academic' 'year' | 'semester'

    grad_query -> intro_grad det noun_grad target_purpose | intro_grad det noun_grad
    intro_grad -> 'i' 'am' 'requesting' | 'i' 'need'
    noun_grad -> 'letter' 'of' 'completion' | 'clearance'
    target_purpose -> 'to' 'facilitate' 'my' 'job' 'application' | 'for' 'jobs'
""")


In [14]:
# We use ChartParser for efficiency with ambiguous grammars
parser = ChartParser(grammar)

# 2. Text Normalization Function
def preprocess_query(text):
    """
    Normalizes text by lowering case and padding punctuation with spaces
    so 'COD,' becomes 'cod ,' allowing NLTK to tokenize it properly.
    """
    text = text.lower()
    # Pad commas and periods with spaces
    text = re.sub(r'([,\.])', r' \1 ', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

# 3. Define 10 varied test queries (9 valid variations, 1 invalid)
test_queries = [
    # Full formal queries
    "Hello COD, I am inquiring about my missing marks for unit CCS 3104",
    "Good morning Dean, what is the deadline for unit registration this semester",

    # Casual/Shortened queries (Testing flexibility)
    "Hi COD, I have missing marks for TOC",
    "Dear Dean, I need a waiver for AI",
    "my results for SMA 2101",

    # Structural variations
    "How do I apply for the bursary for the semester",
    "I need a clearance for jobs",
    "the deadline for special exam now",

    # Grammar tests
    "Dear COD, I am reporting a missing mark in unit SMA 2101",

    # Out of bounds / Fallback test
    "Hello COD, my laptop won't connect to the WiFi"
]

In [15]:
# 4. Evaluate
print(f"{'STATUS':<12} | {'DETECTED INTENT':<20} | {'ORIGINAL QUERY'}")
print("-" * 90)

for query in test_queries:
    tokens = preprocess_query(query)

    try:
        parse_trees = list(parser.parse(tokens))

        if parse_trees:
            tree = parse_trees[0]
            intent_found = False

            # Identify the specific intent from the syntax tree
            for subtree in tree.subtrees():
                label = subtree.label()
                if label in ['marks_query', 'reg_query', 'waiver_query', 'finance_query', 'grad_query']:
                    print(f"[ SUCCESS ]  | {label:<20} | {query}")
                    intent_found = True
                    break

            if not intent_found:
                print(f"[ PARSED ]   | {'Unknown':<20} | {query}")

        else:
            print(f"[ REJECTED ] | {'Route to AI...':<20} | {query}")

    except ValueError as e:
        # Failsafe for words not in our lexicon
        print(f"[ LEX ERROR ]| {'Route to AI...':<20} | {query}")

print("-" * 90)

STATUS       | DETECTED INTENT      | ORIGINAL QUERY
------------------------------------------------------------------------------------------
[ SUCCESS ]  | marks_query          | Hello COD, I am inquiring about my missing marks for unit CCS 3104
[ SUCCESS ]  | reg_query            | Good morning Dean, what is the deadline for unit registration this semester
[ SUCCESS ]  | marks_query          | Hi COD, I have missing marks for TOC
[ SUCCESS ]  | waiver_query         | Dear Dean, I need a waiver for AI
[ SUCCESS ]  | marks_query          | my results for SMA 2101
[ SUCCESS ]  | finance_query        | How do I apply for the bursary for the semester
[ SUCCESS ]  | grad_query           | I need a clearance for jobs
[ SUCCESS ]  | reg_query            | the deadline for special exam now
[ SUCCESS ]  | marks_query          | Dear COD, I am reporting a missing mark in unit SMA 2101
[ LEX ERROR ]| Route to AI...       | Hello COD, my laptop won't connect to the WiFi
------------------------

In [16]:
# cod_responses.py

POLICY_RESPONSES = {
    "marks_query": "Academic Marks & Results: To resolve missing marks or grade corrections, you must fill out the Academic Appeal Form (Form 34A) at the Registrar's office within 14 days of result publication.",

    "reg_query": "Registration & Retakes: The official deadline for unit registration and special exam booking is the 3rd week of the semester. Late registration requires a written letter to the Registrar.",

    "waiver_query": "Registration & Retakes: Prerequisite waivers are only granted to final-year students. You must submit a formal request to the Department Board via the COD's office.",

    "finance_query": "Financial Aid & Bursaries: For University Bursaries, HELB appeals, or fee deferment forms, please visit the Dean of Students' office with your stamped fee statement.",

    "grad_query": "Timetables, Graduation & Careers: Graduation clearance requires you to clear all fee balances and return all library books. Letters of completion are issued 3 weeks after the Senate approves the graduation list.",

    "admin_query": "Administrative & Infrastructure: For physical appointments, inter-university transfers, or reporting lab technical issues, please visit the COD's physical office located in the Beyster Building."
}

In [17]:
# CELL 2: RECOMMENDATION ENGINE

# 1. Map CFG Intents to COD Recommendations
COD_RECOMMENDATIONS = {
    "marks_query": (
        "Category: Academic Marks & Results\n"
        "COD Recommendation: To resolve missing marks or grade corrections, please fill out the "
        "Academic Appeal Form (Form 34A) at the Registrar's office within 14 days of result publication."
    ),
    "reg_query": (
        "Category: Registration & Retakes\n"
        "COD Recommendation: The official deadline for unit registration and special exam booking "
        "is the 3rd week of the semester. Late registration requires a written letter to the Registrar."
    ),
    "waiver_query": (
        "Category: Registration & Retakes (Waivers)\n"
        "COD Recommendation: Prerequisite waivers are generally granted only to final-year students. "
        "Submit a formal written request to the Department Board via the COD's office."
    ),
    "finance_query": (
        "Category: Financial Aid & Bursaries\n"
        "COD Recommendation: For University Bursaries, HELB appeals, or fee deferment forms, "
        "please visit the Dean of Students' office with your stamped fee statement."
    ),
    "grad_query": (
        "Category: Timetables, Graduation & Careers\n"
        "COD Recommendation: Graduation clearance requires you to clear all fee balances and return "
        "all library books. Letters of completion are issued 3 weeks after Senate approval."
    )
}

# 2. The End-to-End Processing Function
def process_student_query(query_text):
    """Parses the text, finds the intent, and returns the COD recommendation."""
    tokens = preprocess_query(query_text)

    try:
        parse_trees = list(parser.parse(tokens))

        if parse_trees:
            tree = parse_trees[0]

            # Extract the specific intent from the syntax tree
            for subtree in tree.subtrees():
                label = subtree.label()
                if label in POLICY_RESPONSES:
                    # Match found! Return the mapped recommendation
                    return POLICY_RESPONSES[label]

            return "Parsed successfully, but no recommendation mapped for this intent."

        else:
            return "CFG Rejected: Query out of bounds. (Action: Route to Hugging Face AI for processing)."

    except ValueError as e:
        return f"Lexical Error (Unknown Word): {e}. (Action: Route to Hugging Face AI for processing)."

print("✅ Recommendation Engine initialized successfully.")

✅ Recommendation Engine initialized successfully.


In [18]:
# CELL 3: RUN TESTS

test_queries = [
    "Hello COD, I am inquiring about my missing marks for unit CCS 3104",
    "Dear Dean, I need a waiver for AI",
    "How do I apply for the bursary for the semester",
    "the deadline for special exam now",
    "Hi COD, my portal is not opening" # This should fail gracefully
]

print("🎓 DeKUT Smart FAQ Assistant - Testing Mode\n" + "="*60)

for i, query in enumerate(test_queries, 1):
    print(f"\nStudent Query {i}: \"{query}\"")
    print("-" * 60)

    recommendation = process_student_query(query)
    print(recommendation)
    print("=" * 60)

🎓 DeKUT Smart FAQ Assistant - Testing Mode

Student Query 1: "Hello COD, I am inquiring about my missing marks for unit CCS 3104"
------------------------------------------------------------
Academic Marks & Results: To resolve missing marks or grade corrections, you must fill out the Academic Appeal Form (Form 34A) at the Registrar's office within 14 days of result publication.

Student Query 2: "Dear Dean, I need a waiver for AI"
------------------------------------------------------------
Registration & Retakes: Prerequisite waivers are only granted to final-year students. You must submit a formal request to the Department Board via the COD's office.

Student Query 3: "How do I apply for the bursary for the semester"
------------------------------------------------------------
Financial Aid & Bursaries: For University Bursaries, HELB appeals, or fee deferment forms, please visit the Dean of Students' office with your stamped fee statement.

Student Query 4: "the deadline for special